
# 01 - Data Preprocessing

This notebook converts the raw sneaker image dataset into a model-ready dataset with a consistent format.

## What this notebook does
1. Reads raw sneaker images from the original dataset directory.
2. Keeps the class/brand subfolder structure.
3. Converts images to RGB.
4. Pads each image to a square canvas with a white background.
5. Resizes each image to `64 x 64`.
6. Saves all processed files to `data/`.

Run this notebook once before training.


In [ ]:
import os
import warnings

# Silence Python-level warnings in notebook output.
warnings.filterwarnings('ignore')

# Reduce noisy runtime warnings from OpenMP/MKL in some environments.
os.environ['KMP_WARNINGS'] = '0'
os.environ['MKL_VERBOSE'] = '0'


import os
from pathlib import Path
from PIL import Image


def resolve_project_root() -> Path:
    # Return repository root when notebook runs from root or notebooks directory.
    cwd = Path.cwd().resolve()
    if (cwd / 'data').exists() or (cwd / 'BayesML_Final_Project.ipynb').exists():
        return cwd
    if (cwd / '..' / 'data').resolve().exists() or (cwd / '..' / 'BayesML_Final_Project.ipynb').resolve().exists():
        return (cwd / '..').resolve()
    return cwd


PROJECT_ROOT = resolve_project_root()
PROJECT_ROOT



## Configure Paths

Set `RAW_DATA_DIR` to the original sneaker dataset location on your machine.

If you already have processed data and only want to regenerate a subset, adjust the output folder or add filtering logic.


In [ ]:

# Update this path to your local raw dataset location.
# Example:
# RAW_DATA_DIR = Path('/path/to/ut-zap50k-images/Shoes/Sneakers and Athletic Shoes')
RAW_DATA_DIR = Path('/Users/alpha/Documents/Learning/Coding/Winter Quarter /Bayes ML/archive/ut-zap50k-images/ut-zap50k-images/Shoes/Sneakers and Athletic Shoes')

PROCESSED_DATA_DIR = PROJECT_ROOT / 'data'
TARGET_IMAGE_SIZE = (64, 64)
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png'}

print(f'Project root: {PROJECT_ROOT}')
print(f'Raw data dir: {RAW_DATA_DIR}')
print(f'Processed dir: {PROCESSED_DATA_DIR}')



## Processing Functions

The helper functions below are intentionally explicit so the preprocessing policy is easy to audit and modify.


In [ ]:

def make_square_with_white_padding(image: Image.Image) -> Image.Image:
    # Pad an RGB image to a square shape using a white background.
    width, height = image.size
    side = max(width, height)

    square = Image.new('RGB', (side, side), color=(255, 255, 255))
    offset_x = (side - width) // 2
    offset_y = (side - height) // 2
    square.paste(image, (offset_x, offset_y))
    return square


def process_and_save_images(source_dir: Path, target_dir: Path) -> int:
    # Process all valid images recursively and preserve relative subfolder structure.
    if not source_dir.exists():
        raise FileNotFoundError(f'Raw data directory not found: {source_dir}')

    target_dir.mkdir(parents=True, exist_ok=True)

    processed_count = 0
    failed_count = 0

    for root, _, files in os.walk(source_dir):
        root_path = Path(root)
        relative_subdir = root_path.relative_to(source_dir)
        target_subdir = target_dir / relative_subdir
        target_subdir.mkdir(parents=True, exist_ok=True)

        for filename in files:
            input_path = root_path / filename
            if input_path.suffix.lower() not in VALID_EXTENSIONS:
                continue

            output_path = target_subdir / filename

            try:
                with Image.open(input_path) as img:
                    rgb_img = img.convert('RGB')
                    squared_img = make_square_with_white_padding(rgb_img)
                    final_img = squared_img.resize(TARGET_IMAGE_SIZE, Image.Resampling.LANCZOS)
                    final_img.save(output_path)
                processed_count += 1

                if processed_count % 2000 == 0:
                    print(f'Processed {processed_count} images...')
            except Exception as exc:
                failed_count += 1
                print(f'Failed to process {input_path}: {exc}')

    print(f'Processing finished. Saved {processed_count} images. Failed: {failed_count}.')
    return processed_count


## Run Preprocessing

In [ ]:

total_processed = process_and_save_images(RAW_DATA_DIR, PROCESSED_DATA_DIR)
print(f'Total processed images: {total_processed}')


## Quick Dataset Summary

In [ ]:

class_dirs = [p for p in PROCESSED_DATA_DIR.iterdir() if p.is_dir()] if PROCESSED_DATA_DIR.exists() else []
image_count = sum(1 for p in PROCESSED_DATA_DIR.rglob('*') if p.suffix.lower() in VALID_EXTENSIONS) if PROCESSED_DATA_DIR.exists() else 0

print(f'Number of classes/brands: {len(class_dirs)}')
print(f'Number of processed images: {image_count}')
